In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from data_collection import build_dataset
from data_selection import select_data, split_data, create_yolo_format, convert_labels_to_numeric_representation
import pandas as pd
from ultralytics import YOLO

Function which pulls and organizes the images. Starts with pulling all the pokemon api, which takes the longest. Followed by kaggle then huggingface. On my machine it takes approximtely 3-4 minutes

In [ ]:
build_dataset()

In [ ]:
df = select_data(sources='all', levels=[1])

In [ ]:
train, test, val = split_data(df)

print(train)

This following code puts the data into a Ultralytics format for yolo models. It essentially tags each image with a bounding box (the whole image is the bounding box) needed for yolo models, and stores it in a text file. The images are also copied to keep referencing in an ultralytics yaml easier. 


In [ ]:
list_of_labels = None # us the follwing list to train for specific pokemon ["magikarp","pikachu", "meowth"]
print((df["label"].value_counts()))

numberic_labels_train = convert_labels_to_numeric_representation(train, list_of_labels)
numberic_labels_test = convert_labels_to_numeric_representation(test, list_of_labels)
numberic_labels_val = convert_labels_to_numeric_representation(val, list_of_labels)

create_yolo_format(train, 'train', numberic_labels_train)
create_yolo_format(test, 'test', numberic_labels_test)
create_yolo_format(val, 'val', numberic_labels_val)

In [ ]:
model = YOLO('yolov8n.pt')
model.train(data='../Dataset/My_yolo_dataset/pokedata.yaml', epochs=20, imgsz=640, device=[4,5], lr0=0.005)



In [ ]:
#This will save the predictions Pokemon-Visualization-Tool/data_operations/runs/detect/predictions/test_predictions
results = model.predict(source="../Dataset/My_yolo_dataset/test/images/", project="predictions/", name="test_predictions", conf=0.25, save=True , exist_ok=True)
